# 05. Проверка длины входа по моделям

Промпт у всех моделей один, а токенов на входе по-разному. Ноутбук показывает, сколько из них приходится на сам текст (разные словари) и сколько добавляет chat template, и печатает шаблон целиком, если он подкладывает свой текст.

Запускается локально (ядро `.venv`), GPU и Colab не нужны: качаются только токенизаторы. Для Llama и Gemma нужен вход в HF (`hf auth login`) с принятыми условиями на странице модели.
Данные берутся из репозитория, результат пишется в `06_pilot/results/prompt_tokens_check.csv`.

In [ ]:
%pip -q install transformers==5.19.0 mistral-common sentencepiece protobuf    # последние две нужны YandexGPT

## Данные и промпт
Скопированы из `05_run_model.ipynb` без изменений: промпт должен совпадать с прогоном.

In [ ]:
import json, random, os
import pandas as pd

SPLITS = ["pilot"]
N = None
K_SHOT = 5
ROOT = ".."             # корень репозитория (ноутбук лежит в 05_protocol)
EDA = f"{ROOT}/03_eda/outputs/dns_cards_500"     # только здесь есть набор pilot
OUT = f"{ROOT}/06_pilot/results"
ATTRS = ["диагональ", "разрешение", "частота", "тип матрицы", "яркость", "контрастность", "изогнутость"]

In [ ]:
cards = {c["url"]: c["description"] for c in json.load(open(f"{ROOT}/data/dns_cards_500.json", encoding="utf-8"))}
pairs = pd.read_csv(f"{EDA}/gold_pairs.csv", dtype=str)
gold_real = {}
for url, g in pairs.groupby("url"):
    gold_real[url] = {a: None for a in ATTRS}          # если атрибута нет в таблице, эталон null
    for _, r in g.iterrows():
        if pd.notna(r["эталон"]):
            gold_real[url][r["атрибут"]] = r["эталон"]
split = json.load(open(f"{EDA}/split_ids.json"))
fewshot = [json.loads(l) for l in open(f"{ROOT}/data/synthetic/fewshot.jsonl", encoding="utf-8")]
shots = random.Random(0).sample(fewshot, K_SHOT)      # одни и те же примеры для всех моделей

def get_items(name):
    if name == "heldout":
        syn = [json.loads(l) for l in open(f"{ROOT}/data/synthetic/heldout.jsonl", encoding="utf-8")]
        items = [{"id": f"heldout{i}", "text": r["description"], "gold": r["gold"]} for i, r in enumerate(syn)]
    else:
        items = [{"id": u, "text": cards[u], "gold": gold_real[u]} for u in split[name]]
    return items[:N] if N else items

ITEMS = {s: get_items(s) for s in SPLITS}
print({s: len(v) for s, v in ITEMS.items()})

In [ ]:
INSTRUCTION = """Извлеки из описания монитора значения семи характеристик и верни JSON.

Правила:
- Бери только то, что прямо написано в тексте описания. Если характеристики в тексте нет, пиши null.
- Не угадывай и не используй свои знания о модели монитора.
- Все значения строки в таком формате: диагональ — число в дюймах ("27"); разрешение — "2560x1440"; частота — число в герцах ("165"); тип матрицы — "IPS", "VA", "TN" или "OLED"; яркость — число в кд/м² ("300"); контрастность — число ("3000", а 1 млн к 1 пиши "1000000"); изогнутость — "да" или "нет".
- Ответ только JSON с ключами: диагональ, разрешение, частота, тип матрицы, яркость, контрастность, изогнутость."""

def make_prompt(text, mode):
    p = INSTRUCTION
    if mode == "few-shot":
        for i, ex in enumerate(shots, 1):
            p += f"\n\nПример {i}.\nОписание: {ex['description']}\nОтвет: {json.dumps(ex['gold'], ensure_ascii=False)}"
    return p + f"\n\nОписание: {text}\nОтвет:"

SCHEMA_JSON = {"type": "object",
               "properties": {a: {"type": ["string", "null"]} for a in ATTRS},
               "required": ATTRS, "additionalProperties": False}

## Проверка

In [ ]:
"""Проверка: откуда у моделей разная длина входа и нет ли там лишнего.

Промпт у всех один, но токенов на входе по-разному (Ministral и GigaChat в zero-shot в 2–3 раза больше остальных).
Длину делим на две части:
  - «текст»   — сам промпт, токенизатором модели без шаблона; разница тут — это просто разные словари (русский режется по-разному);
  - «обёртка» — что добавляет chat template сверх промпта (служебные токены, системный промпт по умолчанию, дата и т.п.).
Обёртка в несколько токенов — норма. Сотни токенов значит, что шаблон подкладывает свой текст: его надо увидеть
и решить, оставлять ли (модель получает не тот вход, что остальные).

GPU не нужен. Результат: <OUT>/prompt_tokens_check.csv (числа) и <OUT>/chat_templates/<модель>.txt (вход целиком).
"""
import json, os

import pandas as pd
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

TOK_MODELS = {      # все восемь, включая GigaChat (в MODELS ноутбука он может быть закомментирован)
    "qwen35_9b":       "Qwen/Qwen3.5-9B",
    "gemma4_e4b":      "google/gemma-4-E4B-it",
    "llama31_8b":      "meta-llama/Llama-3.1-8B-Instruct",
    "ministral3_8b":   "mistralai/Ministral-3-8B-Instruct-2512-BF16",
    "phi4_mini":       "microsoft/Phi-4-mini-instruct",
    "gigachat31_10b":  "ai-sage/GigaChat3.1-10B-A1.8B-bf16",
    "tlite21_8b":      "t-tech/T-lite-it-2.1",
    "yandex5_lite_8b": "yandex/YandexGPT-5-Lite-8B-instruct",
}
MAX_WRAP = 50       # обёртка длиннее — показываем, что в ней
MAX_RATIO = 1.3     # текст длиннее медианы по моделям во столько раз — словарь заметно хуже режет русский
MARK = "<<ПРОМПТ>>"


def load_tok(key, name):
    """как в ячейке загрузки модели; у GigaChat правим config.json, веса не качаем"""
    if key == "gigachat31_10b":
        src = snapshot_download(name, allow_patterns=["*.json", "*.model", "*.txt", "*.py", "*.jinja"])
        fixed = os.path.abspath("gigachat_tok")      # рядом с ноутбуком, только ссылки и config.json
        os.makedirs(fixed, exist_ok=True)
        for f in os.listdir(src):
            if f != "config.json" and not os.path.exists(f"{fixed}/{f}"):
                os.symlink(os.path.realpath(f"{src}/{f}"), f"{fixed}/{f}")
        cfg = json.load(open(f"{src}/config.json"))
        cfg["routed_scaling_factor"] = float(cfg["routed_scaling_factor"])
        json.dump(cfg, open(f"{fixed}/config.json", "w"), indent=2)
        name = fixed
    kw = {"fix_mistral_regex": True} if key == "ministral3_8b" else {}
    return AutoTokenizer.from_pretrained(name, **kw)


def wrap(tok, content):
    """строка входа так же, как в generate"""
    chat = [{"role": "user", "content": content}]
    return tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True, enable_thinking=False)


def ntok(tok, s):
    return len(tok(s, add_special_tokens=False)["input_ids"])


pilot = get_items("pilot")
rows, templates = [], {}
for key, name in TOK_MODELS.items():
    try:
        tok = load_tok(key, name)
        templates[key] = wrap(tok, MARK)
        os.makedirs(f"{OUT}/chat_templates", exist_ok=True)      # вход целиком для каждой модели: <<ПРОМПТ>> = наш промпт
        with open(f"{OUT}/chat_templates/{key}.txt", "w", encoding="utf-8") as f:
            f.write(templates[key])
        for mode in ("zero-shot", "few-shot"):
            full, text = [], []
            for it in pilot:
                p = make_prompt(it["text"], mode)
                full.append(ntok(tok, wrap(tok, p)))
                text.append(ntok(tok, p))
            ids = tok(wrap(tok, make_prompt(pilot[0]["text"], mode)), add_special_tokens=False)["input_ids"]
            rows.append({"model": key, "mode": mode, "n": len(full),
                         "всего": round(sum(full) / len(full), 1),
                         "текст": round(sum(text) / len(text), 1),
                         "обёртка": round((sum(full) - sum(text)) / len(full), 1),
                         "BOS_всего": ids.count(tok.bos_token_id) if tok.bos_token_id is not None else 0})
        print(key, "готово")
    except Exception as e:
        print(key, "ОШИБКА:", repr(e))

path = f"{OUT}/prompt_tokens_check.csv"
df = pd.DataFrame(rows)
try:                    # пересчитанные модели заменяем, строки остальных (например, упавших сейчас) оставляем
    old = pd.read_csv(path)
    df = pd.concat([old[~old["model"].isin(df["model"])], df], ignore_index=True) if len(df) else old
except (FileNotFoundError, pd.errors.EmptyDataError):
    pass
order = list(TOK_MODELS)
df = df.sort_values("model", key=lambda m: m.map(order.index), kind="stable")
df["текст_к_медиане"] = (df["текст"] / df.groupby("mode")["текст"].transform("median")).round(2)   # медиана по всем моделям в файле
df.to_csv(path, index=False)
print(df.to_string(index=False))

print("\nЧто не так:")
bad = False
for key in [k for k in df["model"].unique() if k in templates]:     # только посчитанные в этот запуск
    d = df[df["model"] == key].set_index("mode")
    wrap_len = d["обёртка"].max()
    if wrap_len > MAX_WRAP:
        bad = True
        print(f"\n[{key}] шаблон добавляет ~{wrap_len:.0f} токенов сверх промпта. Вход целиком ({MARK} = наш промпт):")
        print("-" * 80 + "\n" + templates[key] + "\n" + "-" * 80)
    if d["текст_к_медиане"].max() > MAX_RATIO:
        bad = True
        print(f"[{key}] сам промпт в {d['текст_к_медиане'].max()}× длиннее медианы: словарь хуже режет русский (это не ошибка, но влияет на время)")
    if d["BOS_всего"].max() > 1:
        bad = True
        print(f"[{key}] BOS во входе {d['BOS_всего'].max()} раза: шаблон и токенизатор оба его ставят")
    if abs(d.loc["few-shot", "обёртка"] - d.loc["zero-shot", "обёртка"]) > 5:
        bad = True
        print(f"[{key}] обёртка в zero-shot и few-shot разная ({d.loc['zero-shot', 'обёртка']} и {d.loc['few-shot', 'обёртка']}): шаблон что-то меняет в тексте")
if not bad:
    print("ничего: разница только в словарях, шаблоны добавляют лишь служебные токены")
